# step3_features

**Owner:** P4  
**Purpose:** Feature tables and text/entity embeddings.  
**Inputs:** `data/processed/` from step 1  
**Outputs:** `data/processed/features_*.parquet`, `news_embeddings.npy`

> Runtime: VS Code/Jupyter locally, or Colab with the repository
and data configured. Run cells top to bottom; keep outputs saved.

In [2]:
# --- Setup: run this first, every session ---
import sys
from pathlib import Path

# Find the repository root, whether the notebook starts
# in the main folder or inside step3_features/
current_folder = Path.cwd().resolve()

PROJECT_ROOT = next(
    (
        folder
        for folder in [current_folder, *current_folder.parents]
        if (folder / "config.py").is_file()
        and (folder / "utils").is_dir()
    ),
    None,
)

if PROJECT_ROOT is None:
    raise FileNotFoundError(
        "Cannot find config.py and utils/. "
        "Open the cloned project folder in VS Code."
    )

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

import numpy as np
import pandas as pd
import config

from utils.data_utils import (
    load_news,
    load_behaviors,
    explode_impressions,
)

from utils.metrics import (
    evaluate,
    evaluate_by_segment,
    topk,
    topk_diversity,
)

print("Python:", sys.executable)
print("Repository:", PROJECT_ROOT)
print("Config loaded from:", config.__file__)
print("Data folder:", config.DATA_DIR)

Python: c:\Users\eishw\OneDrive - National University of Singapore\BT4222\Project\BT4222-Project-Group-4--MIND\.venv\Scripts\python.exe
Repository: C:\Users\eishw\OneDrive - National University of Singapore\BT4222\Project\BT4222-Project-Group-4--MIND
Config loaded from: C:\Users\eishw\OneDrive - National University of Singapore\BT4222\Project\BT4222-Project-Group-4--MIND\config.py
Data folder: C:\Users\eishw\OneDrive - National University of Singapore\BT4222\Project\BT4222-Project-Group-4--MIND\data


# Feature Engineering Overview

This notebook prepares the features used by the Feature-Based MLP news recommendation model.

The objective is to transform the raw MIND-large candidate and news data into structured features that represent:

- User identity and user preferences
- Article identity
- Article category and subcategory
- Article exposure history
- Whether an article is cold or warm
- Time-related context of the recommendation

The resulting features are later consumed by the Feature-Based MLP in `step3_features_mlp`.

## Feature Groups

| Feature | Type | Purpose |
|---|---|---|
| User ID | Categorical | Represents the user receiving the recommendation |
| News ID | Categorical | Represents the candidate article |
| Category | Categorical | Captures the broad topic of the article |
| Subcategory | Categorical | Captures the more specific topic |
| Prior exposure count | Numerical | Represents how frequently the article has previously been exposed |
| Cold article | Binary | Identifies articles with low prior exposure |
| Hour of day | Cyclical | Captures time-of-day effects on user behaviour |
| Day of week | Cyclical | Captures weekly usage patterns |

Categorical features are later represented using embeddings, while numerical and binary features are transformed into numerical values suitable for the neural network.

The feature engineering process is designed to avoid data leakage by using information available before the recommendation event.

# Load News Metadata

The MIND dataset provides metadata for each news article, including its category, subcategory, title, abstract, and entity information.

For the Feature-Based MLP, the main metadata features used are the article's `category` and `subcategory`.

The article title and abstract are not directly used as text features in this model. This keeps the model as a feature-based baseline rather than a text-based neural recommender such as NRMS.

In [4]:
# --- Load processed news metadata ---

from pathlib import Path
import pyarrow.parquet as pq

PROCESSED_DIR = Path(
    r"G:\.shortcut-targets-by-id\1D4Db1s7DJ5PVK9CFiV-pklpCW5dr1uOM\BT4222_Project\data\processed"
)

news_metadata_path = PROCESSED_DIR / "news_metadata.parquet"

if not news_metadata_path.exists():
    raise FileNotFoundError(
        f"Cannot find news metadata file:\n{news_metadata_path}"
    )

news_metadata = pd.read_parquet(news_metadata_path)

print("News metadata shape:", news_metadata.shape)
print("\nColumns:")
print(news_metadata.columns.tolist())

print("\nSample:")
display(news_metadata.head())

News metadata shape: (104151, 9)

Columns:
['news_id', 'category', 'subcategory', 'title', 'abstract', 'url', 'title_entities', 'abstract_entities', 'news_idx']

Sample:


,news_id,category,subcategory,title,abstract,url,title_entities,abstract_entities,news_idx
0,N88753,lifestyle,lifestyleroyals,"The Brands Queen Elizabeth, Prince Charles, an...","Shop the notebooks, jackets, and more that the...",https://assets.msn.com/labs/mind/AAGH0ET.html,"[{""Label"": ""Prince Philip, Duke of Edinburgh"",...",[],1
1,N45436,news,newsscienceandtechnology,Walmart Slashes Prices on Last-Generation iPads,Apple's new iPad releases bring big deals on l...,https://assets.msn.com/labs/mind/AABmf2I.html,"[{""Label"": ""IPad"", ""Type"": ""J"", ""WikidataId"": ...","[{""Label"": ""IPad"", ""Type"": ""J"", ""WikidataId"": ...",2
2,N23144,health,weightloss,50 Worst Habits For Belly Fat,These seemingly harmless habits are holding yo...,https://assets.msn.com/labs/mind/AAB19MK.html,"[{""Label"": ""Adipose tissue"", ""Type"": ""C"", ""Wik...","[{""Label"": ""Adipose tissue"", ""Type"": ""C"", ""Wik...",3
3,N86255,health,medical,Dispose of unwanted prescription drugs during ...,,https://assets.msn.com/labs/mind/AAISxPN.html,"[{""Label"": ""Drug Enforcement Administration"", ...",[],4
4,N93187,news,newsworld,The Cost of Trump's Aid Freeze in the Trenches...,Lt. Ivan Molchanets peeked over a parapet of s...,https://assets.msn.com/labs/mind/AAJgNsz.html,[],"[{""Label"": ""Ukraine"", ""Type"": ""G"", ""WikidataId...",5


# Build Category and Subcategory Vocabularies

The article category and subcategory are categorical variables and cannot be passed directly into the neural network as text.

We therefore create integer vocabularies for both features.

Index `0` is reserved for unknown or missing values. This allows the model to handle categories that are not present in the vocabulary without producing invalid embedding indices.

The resulting integer IDs are later converted into embedding representations by the Feature-Based MLP.

In [5]:
# --- Build category and subcategory vocabularies ---

# Reserve index 0 for unknown / missing values
category_values = sorted(
    news_metadata["category"]
    .astype(str)
    .loc[news_metadata["category"].astype(str).str.len() > 0]
    .unique()
)

subcategory_values = sorted(
    news_metadata["subcategory"]
    .astype(str)
    .loc[news_metadata["subcategory"].astype(str).str.len() > 0]
    .unique()
)

category_to_idx = {
    category: idx
    for idx, category in enumerate(category_values, start=1)
}

subcategory_to_idx = {
    subcategory: idx
    for idx, subcategory in enumerate(subcategory_values, start=1)
}

NUM_CATEGORIES = len(category_to_idx) + 1
NUM_SUBCATEGORIES = len(subcategory_to_idx) + 1

print("Number of categories:", len(category_to_idx))
print("Category embedding size:", NUM_CATEGORIES)

print("\nNumber of subcategories:", len(subcategory_to_idx))
print("Subcategory embedding size:", NUM_SUBCATEGORIES)

print("\nCategory mapping sample:")
print(list(category_to_idx.items())[:10])

print("\nSubcategory mapping sample:")
print(list(subcategory_to_idx.items())[:10])

Number of categories: 18
Category embedding size: 19

Number of subcategories: 285
Subcategory embedding size: 286

Category mapping sample:
[('autos', 1), ('entertainment', 2), ('finance', 3), ('foodanddrink', 4), ('games', 5), ('health', 6), ('kids', 7), ('lifestyle', 8), ('middleeast', 9), ('movies', 10)]

Subcategory mapping sample:
[('ads-latingrammys', 1), ('ads-lung-health', 2), ('advice', 3), ('animals', 4), ('autosbuying', 5), ('autoscartech', 6), ('autosclassics', 7), ('autoscompact', 8), ('autosconvertibles', 9), ('autosenthusiasts', 10)]


## Validate Categorical Mappings

The vocabulary mappings are checked by converting the article metadata into integer IDs.

A valid article category or subcategory should receive a positive integer ID, while an unknown or missing value should be represented by `0`.

In [6]:
# --- Validate categorical mappings ---

news_metadata["category_idx"] = (
    news_metadata["category"]
    .astype(str)
    .map(category_to_idx)
    .fillna(0)
    .astype(np.int64)
)

news_metadata["subcategory_idx"] = (
    news_metadata["subcategory"]
    .astype(str)
    .map(subcategory_to_idx)
    .fillna(0)
    .astype(np.int64)
)

print("Category ID range:")
print(
    "min =", news_metadata["category_idx"].min(),
    "| max =", news_metadata["category_idx"].max()
)

print("\nSubcategory ID range:")
print(
    "min =", news_metadata["subcategory_idx"].min(),
    "| max =", news_metadata["subcategory_idx"].max()
)

print("\nMapped sample:")
display(
    news_metadata[
        [
            "news_id",
            "category",
            "category_idx",
            "subcategory",
            "subcategory_idx",
        ]
    ].head(10)
)

Category ID range:
min = 1 | max = 18

Subcategory ID range:
min = 1 | max = 285

Mapped sample:


,news_id,category,category_idx,subcategory,subcategory_idx
0,N88753,lifestyle,8,lifestyleroyals,151
1,N45436,news,12,newsscienceandtechnology,197
2,N23144,health,6,weightloss,281
3,N86255,health,6,medical,161
4,N93187,news,12,newsworld,205
5,N75236,health,6,voices,276
6,N99744,health,6,medical,161
7,N5771,health,6,cardio,44
8,N124534,sports,14,football_nfl,104
9,N51947,news,12,newsscienceandtechnology,197


# Build User and Article ID Vocabularies

User IDs and article IDs are categorical identifiers that cannot be passed directly into the neural network.

For the Feature-Based MLP, these IDs are converted into integer indices that are later used by embedding layers.

The user and article vocabularies are constructed from the FIT training data only. This prevents information from the TUNE or EVAL periods from influencing the model's training representation.

Index `0` is reserved for unknown users or articles.

In [7]:
# --- Build FIT-only user and article vocabularies ---

import pyarrow.parquet as pq

fit_path = PROCESSED_DIR / "fit_candidates_sampled.parquet"

if not fit_path.exists():
    raise FileNotFoundError(
        f"Cannot find FIT candidates file:\n{fit_path}"
    )

fit_parquet = pq.ParquetFile(fit_path)

print("FIT rows:", fit_parquet.metadata.num_rows)
print("FIT row groups:", fit_parquet.num_row_groups)

FIT rows: 12894210
FIT row groups: 133


In [8]:
# --- Collect unique FIT user and article IDs without loading the full file ---

fit_users = set()
fit_news = set()

for arrow_batch in fit_parquet.iter_batches(
    batch_size=100_000,
    columns=["user_idx", "news_idx"]
):
    batch_df = arrow_batch.to_pandas()

    fit_users.update(
        batch_df["user_idx"].dropna().astype(np.int64).unique()
    )

    fit_news.update(
        batch_df["news_idx"].dropna().astype(np.int64).unique()
    )

    del batch_df
    del arrow_batch

print("Unique FIT users:", len(fit_users))
print("Unique FIT articles:", len(fit_news))

Unique FIT users: 654870
Unique FIT articles: 19090


In [9]:
# --- Prepare known-ID sets and embedding dimensions ---

known_user_ids = set(fit_users)
known_news_ids = set(fit_news)

NUM_USERS = max(known_user_ids) + 1
NUM_NEWS = max(known_news_ids) + 1

print("User embedding size:", NUM_USERS)
print("News embedding size:", NUM_NEWS)
print("Unknown user index: 0")
print("Unknown news index: 0")

User embedding size: 711223
News embedding size: 101527
Unknown user index: 0
Unknown news index: 0


# Define Feature Transformation

This section converts candidate-level data into the numerical inputs required by the Feature-Based MLP.

The transformation includes:

- User and article IDs
- Category and subcategory IDs
- Log-transformed prior exposure
- Cold article indicator
- Cyclical hour-of-day features
- Cyclical day-of-week features
- Click labels

Unknown users and articles are mapped to index `0`.

Time variables are represented using sine and cosine transformations so that cyclical relationships are preserved. For example, 23:00 and 00:00 are treated as close together rather than being treated as distant numerical values.

In [10]:
# --- Transform candidate rows into MLP-ready features ---

def transform_batch(batch_df):

    # User and article IDs
    user_ids = batch_df["user_idx"].to_numpy(dtype=np.int64).copy()
    news_ids = batch_df["news_idx"].to_numpy(dtype=np.int64).copy()

    # Map unknown IDs to 0
    user_ids[~np.isin(user_ids, known_user_ids)] = 0
    news_ids[~np.isin(news_ids, known_news_ids)] = 0

    # Category IDs
    category_ids = (
        batch_df["category"]
        .astype(str)
        .map(category_to_idx)
        .fillna(0)
        .to_numpy(dtype=np.int64)
    )

    # Subcategory IDs
    subcategory_ids = (
        batch_df["subcategory"]
        .astype(str)
        .map(subcategory_to_idx)
        .fillna(0)
        .to_numpy(dtype=np.int64)
    )

    # Exposure and cold-article features
    exposure = np.log1p(
        batch_df["prior_exposure_count"].to_numpy(dtype=np.float32)
    )

    cold_article = batch_df[
        "cold_article"
    ].to_numpy(dtype=np.float32)

    # Time features
    timestamps = pd.to_datetime(batch_df["time"])

    hour = timestamps.dt.hour.to_numpy(dtype=np.float32)
    day = timestamps.dt.day_of_week.to_numpy(dtype=np.float32)

    # Cyclical hour representation
    hour_sin = np.sin(2 * np.pi * hour / 24)
    hour_cos = np.cos(2 * np.pi * hour / 24)

    # Cyclical day-of-week representation
    day_sin = np.sin(2 * np.pi * day / 7)
    day_cos = np.cos(2 * np.pi * day / 7)

    # Combine numerical features
    numerical_features = np.column_stack([
        exposure,
        cold_article,
        hour_sin,
        hour_cos,
        day_sin,
        day_cos
    ]).astype(np.float32)

    return {
        "user_ids": user_ids,
        "news_ids": news_ids,
        "category_ids": category_ids,
        "subcategory_ids": subcategory_ids,
        "numerical_features": numerical_features,
        "labels": batch_df["label"].to_numpy(
            dtype=np.float32
        )
    }

# Validate Feature Transformation

A small sample from the FIT data is transformed to verify that all expected features are generated correctly.

This validation does not train the model or modify the FIT dataset. It only checks the structure, data types, ranges, and dimensionality of the engineered features.

In [11]:
# --- Validate feature transformation on a small FIT sample ---

sample_arrow = next(
    fit_parquet.iter_batches(
        batch_size=1000,
        columns=[
            "user_idx",
            "news_idx",
            "category",
            "subcategory",
            "prior_exposure_count",
            "cold_article",
            "time",
            "label"
        ]
    )
)

sample_df = sample_arrow.to_pandas()

sample_features = transform_batch(sample_df)

print("Feature keys:")
print(list(sample_features.keys()))

print("\nNumber of rows:", len(sample_features["labels"]))

print("\nShapes:")
for name, values in sample_features.items():
    print(f"{name}: {values.shape}")

print("\nData types:")
for name, values in sample_features.items():
    print(f"{name}: {values.dtype}")

Feature keys:
['user_ids', 'news_ids', 'category_ids', 'subcategory_ids', 'numerical_features', 'labels']

Number of rows: 1000

Shapes:
user_ids: (1000,)
news_ids: (1000,)
category_ids: (1000,)
subcategory_ids: (1000,)
numerical_features: (1000, 6)
labels: (1000,)

Data types:
user_ids: int64
news_ids: int64
category_ids: int64
subcategory_ids: int64
numerical_features: float32
labels: float32


# Inspect Numerical Feature Ranges

The numerical features are inspected to ensure that their values are reasonable before being passed to the neural network.

Prior exposure is log-transformed using `log1p` to reduce the effect of highly skewed exposure counts.

The remaining features are bounded cyclical or binary variables.

In [12]:
# --- Inspect numerical feature ranges ---

numerical = sample_features["numerical_features"]

numerical_summary = pd.DataFrame(
    numerical,
    columns=[
        "log_exposure",
        "cold_article",
        "hour_sin",
        "hour_cos",
        "day_sin",
        "day_cos"
    ]
).describe().T

display(numerical_summary)

,count,mean,std,min,25%,50%,75%,max
log_exposure,1000.0,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
cold_article,1000.0,1.000000,0.000000,1.000000,1.000000,1.000000,1.000000,1.000000
hour_sin,1000.0,0.443660,0.266491,0.000000,0.258819,0.500000,0.707107,0.707107
hour_cos,1000.0,0.847384,0.119053,0.707107,0.707107,0.866025,0.965926,1.000000
day_sin,1000.0,-0.974928,0.000000,-0.974928,-0.974928,-0.974928,-0.974928,-0.974928
day_cos,1000.0,-0.222521,0.000000,-0.222521,-0.222521,-0.222521,-0.222521,-0.222521


# Check for Missing or Invalid Feature Values

The engineered features are checked for missing or invalid numerical values.

Unknown categorical values are represented using index `0`, while the numerical feature matrix should not contain NaN or infinite values.

In [13]:
# --- Check for invalid feature values ---

print("Missing values in numerical features:")
print(np.isnan(numerical).sum())

print("\nInfinite values in numerical features:")
print(np.isinf(numerical).sum())

print("\nUnknown category IDs (0):",
      (sample_features["category_ids"] == 0).sum())

print("Unknown subcategory IDs (0):",
      (sample_features["subcategory_ids"] == 0).sum())

print("Unknown user IDs (0):",
      (sample_features["user_ids"] == 0).sum())

print("Unknown news IDs (0):",
      (sample_features["news_ids"] == 0).sum())

Missing values in numerical features:
0

Infinite values in numerical features:
0

Unknown category IDs (0): 0
Unknown subcategory IDs (0): 0
Unknown user IDs (0): 1000
Unknown news IDs (0): 1000


# Feature Engineering Summary

The feature engineering pipeline produces four categorical feature groups and six numerical/context features for the Feature-Based MLP.

## Categorical Features

- User ID
- News ID
- Category
- Subcategory

These features are represented using embedding layers in the MLP.

## Numerical and Context Features

- Log-transformed prior exposure count
- Cold article indicator
- Hour-of-day sine
- Hour-of-day cosine
- Day-of-week sine
- Day-of-week cosine

The final MLP input therefore consists of:

- 32 dimensions from the user embedding
- 32 dimensions from the news embedding
- 8 dimensions from the category embedding
- 16 dimensions from the subcategory embedding
- 6 numerical/context features

This results in a **94-dimensional feature representation** for each user-article candidate.

The engineered features are then passed to the Feature-Based MLP for click-probability prediction.

In [14]:
# --- Final feature summary ---

feature_summary = pd.DataFrame({
    "Feature": [
        "User ID",
        "News ID",
        "Category",
        "Subcategory",
        "Log prior exposure",
        "Cold article",
        "Hour sin/cos",
        "Day sin/cos"
    ],
    "Type": [
        "Categorical",
        "Categorical",
        "Categorical",
        "Categorical",
        "Numerical",
        "Binary",
        "Cyclical",
        "Cyclical"
    ],
    "MLP representation": [
        "32-d embedding",
        "32-d embedding",
        "8-d embedding",
        "16-d embedding",
        "1 numerical value",
        "1 numerical value",
        "2 numerical values",
        "2 numerical values"
    ],
    "Dimensions": [
        32,
        32,
        8,
        16,
        1,
        1,
        2,
        2
    ]
})

display(feature_summary)

print("Total MLP input dimensions: 94")

,Feature,Type,MLP representation,Dimensions
0,User ID,Categorical,32-d embedding,32
1,News ID,Categorical,32-d embedding,32
2,Category,Categorical,8-d embedding,8
3,Subcategory,Categorical,16-d embedding,16
4,Log prior exposure,Numerical,1 numerical value,1
5,Cold article,Binary,1 numerical value,1
6,Hour sin/cos,Cyclical,2 numerical values,2
7,Day sin/cos,Cyclical,2 numerical values,2


Total MLP input dimensions: 94


In [15]:
# --- Verify P1 processed data files ---

expected_p1_files = [
    "fit_candidates_sampled.parquet",
    "tune_candidates.parquet",
    "eval_candidates.parquet",
    "news_metadata.parquet",
]

print("Processed data directory:")
print(PROCESSED_DIR)

print("\nP1 processed files:")

for filename in expected_p1_files:
    filepath = PROCESSED_DIR / filename

    if filepath.exists():
        print(f"✓ {filename}")
    else:
        print(f"✗ MISSING: {filename}")

Processed data directory:
G:\.shortcut-targets-by-id\1D4Db1s7DJ5PVK9CFiV-pklpCW5dr1uOM\BT4222_Project\data\processed

P1 processed files:
✓ fit_candidates_sampled.parquet
✓ tune_candidates.parquet
✓ eval_candidates.parquet
✓ news_metadata.parquet


In [16]:
# --- Verify processed dataset sizes ---

for filename in expected_p1_files:
    filepath = PROCESSED_DIR / filename
    parquet_file = pq.ParquetFile(filepath)

    print(
        f"{filename}: "
        f"{parquet_file.metadata.num_rows:,} rows, "
        f"{parquet_file.num_row_groups} row groups"
    )

fit_candidates_sampled.parquet: 12,894,210 rows, 133 row groups
tune_candidates.parquet: 17,400,106 rows, 35 row groups
eval_candidates.parquet: 14,085,557 rows, 29 row groups
news_metadata.parquet: 104,151 rows, 1 row groups
